# 06 — Claude Teacher: High-Quality MCQ Data Generation

**Why this notebook?**
Qwen2.5-7B generates acceptable MCQs but can produce:
- Weak distractors (obviously wrong)
- English-only questions for Thai exam PDFs
- Uneven difficulty (mostly medium)

**Solution**: Use `claude-opus-4-6` (Anthropic API) as a stronger teacher to generate a gold-standard training set.
Claude then feeds this into the fine-tuning pipeline (notebook 04).

**Requirements:**
- `ANTHROPIC_API_KEY` in `.env` or environment
- PDFs in `../dataset/sample_pdfs/` (or point `PDF_DIR` to your folder)
- `pip install anthropic pdfplumber`

> Acknowledgement: Computational resources supported by KU Nontri AI, Kasetsart University

In [ ]:
import json, os, re, time
from pathlib import Path
from dotenv import load_dotenv

import anthropic
import pdfplumber

load_dotenv()

# ── Config ────────────────────────────────────────────────────────────────
PDF_DIR    = Path('../dataset/sample_pdfs')   # input PDFs
OUT_DIR    = Path('../dataset/rated')          # output (same schema as rated/)
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL      = 'claude-opus-4-6'
MAX_CHARS  = 6000   # chars per PDF chunk sent to Claude (~4K tokens)
QS_PER_GAME = 10    # questions per (PDF, game_type) pair
GAME_TYPES = ['flappy', 'racer', 'shooter', 'snake', 'bricks']

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
print(f'✅ Anthropic client ready | model: {MODEL}')
print(f'📂 PDFs dir   : {PDF_DIR}')
print(f'💾 Output dir : {OUT_DIR}')

In [ ]:
# ── PDF helpers ──────────────────────────────────────────────────────────

def extract_pdf_text(pdf_path: Path, max_chars: int = MAX_CHARS) -> str:
    """Extract text from PDF, truncate to max_chars for API context."""
    with pdfplumber.open(pdf_path) as pdf:
        pages = [p.extract_text() or '' for p in pdf.pages]
    full_text = '\n'.join(pages)
    return full_text[:max_chars]

# Preview PDFs available
pdfs = sorted(PDF_DIR.glob('*.pdf'))
print(f'Found {len(pdfs)} PDF(s):')
for p in pdfs:
    print(f'  • {p.name}')
if not pdfs:
    print('⚠️  No PDFs found — add .pdf files to', PDF_DIR)

In [ ]:
# ── Prompt builder ───────────────────────────────────────────────────────

GAME_HINTS = {
    'flappy':  'ให้ทุกคำถามสั้นมาก ไม่เกิน 12 คำ และตัวเลือกแต่ละข้อ 1-4 คำเท่านั้น',
    'racer':   'ให้ทุกคำถามสั้น ไม่เกิน 12 คำ ตัวเลือก 1-4 คำ เหมาะกับการอ่านขณะขับรถ',
    'shooter': 'เน้นคำถามแบบ "คำศัพท์ใดหมายความว่า..." หรือ "ข้อใดคือนิยามของ..."',
    'snake':   'เน้นคำถามลำดับขั้นตอน เช่น "ขั้นตอนแรกของ... คืออะไร"',
    'bricks':  'เน้นคำถามนิยาม เช่น "X คืออะไร" หรือ "X หมายถึงอะไร" ทดสอบคำศัพท์',
}

def build_prompt(source_text: str, game_type: str) -> str:
    hint = GAME_HINTS[game_type]
    return f"""คุณคือผู้เชี่ยวชาญด้านการออกข้อสอบมหาวิทยาลัยไทย

{hint}

สร้างคำถามปรนัย (multiple-choice) จำนวนพอดี {QS_PER_GAME} ข้อ จากเนื้อหาด้านล่าง
**กฎสำคัญ:**
- คำถามและตัวเลือกต้องมาจากเนื้อหาที่ให้เท่านั้น ห้ามแต่งใหม่
- สัดส่วนความยาก: 3 ง่าย (difficulty=1) / 5 ปานกลาง (difficulty=2) / 2 ยาก (difficulty=3)
- ตัวเลือก A B C D ต้องสลับสับเปลี่ยนว่าข้อใดถูก (กระจายสม่ำเสมอ ไม่ให้ A ถูกทุกข้อ)
- ส่งออกเป็น JSON array เท่านั้น ไม่มีข้อความอื่น ไม่มี markdown code fence

รูปแบบแต่ละข้อ:
{{
  "id": 1,
  "question": "...",
  "choices": ["A. ...", "B. ...", "C. ...", "D. ..."],
  "correct": 0,
  "explanation": "อธิบายสั้นๆ ว่าทำไมคำตอบนี้ถูก",
  "difficulty": 1,
  "difficulty_reason": "อธิบายระดับความยาก"
}}

เนื้อหา:
{source_text}"""

print('✅ Prompt builder ready')

In [ ]:
# ── Claude API call ──────────────────────────────────────────────────────

def generate_questions(source_text: str, game_type: str, source_doc: str) -> list[dict]:
    """Call claude-opus-4-6 to generate MCQ questions for one (PDF, game_type) pair."""
    prompt = build_prompt(source_text, game_type)
    
    # Use adaptive thinking for highest quality output
    response = client.messages.create(
        model=MODEL,
        max_tokens=4096,
        thinking={"type": "adaptive"},
        messages=[{"role": "user", "content": prompt}],
    )
    
    # Extract text (skip thinking blocks)
    raw = ''
    for block in response.content:
        if block.type == 'text':
            raw = block.text
            break
    
    # Parse JSON
    # Strip markdown fences if Claude added them anyway
    raw = re.sub(r'^```json\s*', '', raw.strip(), flags=re.MULTILINE)
    raw = re.sub(r'^```\s*$', '', raw.strip(), flags=re.MULTILINE)
    raw = raw.strip()
    
    questions = json.loads(raw)
    
    # Stamp game_type, source_doc, difficulty_teacher on each question
    for i, q in enumerate(questions):
        q['id'] = i + 1
        q['game_type'] = game_type
        q['source_doc'] = source_doc
        q['difficulty_teacher'] = q.get('difficulty', 2)
    
    return questions

print('✅ generate_questions() ready')

In [ ]:
# ── Main generation loop ─────────────────────────────────────────────────

def already_done(pdf_name: str, game_type: str) -> bool:
    out_file = OUT_DIR / f'{Path(pdf_name).stem}_{game_type}.json'
    return out_file.exists()

errors = []
total_generated = 0

for pdf_path in sorted(pdfs):
    source_text = extract_pdf_text(pdf_path)
    if not source_text.strip():
        print(f'⚠️  {pdf_path.name}: no text extracted (scanned PDF?), skipping')
        continue
    
    print(f'\n📄 {pdf_path.name} ({len(source_text):,} chars)')
    
    for game_type in GAME_TYPES:
        if already_done(pdf_path.name, game_type):
            print(f'  ✅ {game_type} — already exists, skipping')
            continue
        
        try:
            print(f'  🤖 {game_type} — generating {QS_PER_GAME} questions ...')
            questions = generate_questions(source_text, game_type, pdf_path.name)
            
            out_file = OUT_DIR / f'{pdf_path.stem}_{game_type}.json'
            with open(out_file, 'w', encoding='utf-8') as f:
                json.dump(questions, f, ensure_ascii=False, indent=2)
            
            total_generated += len(questions)
            print(f'  ✅ {game_type} — {len(questions)} questions saved → {out_file.name}')
            
            # Small delay to respect rate limits
            time.sleep(1)
            
        except json.JSONDecodeError as e:
            print(f'  ❌ {game_type} — JSON parse error: {e}')
            errors.append((pdf_path.name, game_type, str(e)))
        except anthropic.RateLimitError:
            print(f'  ⏳ {game_type} — rate limited, waiting 60s ...')
            time.sleep(60)
            errors.append((pdf_path.name, game_type, 'rate_limit'))
        except Exception as e:
            print(f'  ❌ {game_type} — error: {e}')
            errors.append((pdf_path.name, game_type, str(e)))

print(f'\n============================================================')
print(f' Done! Total questions generated: {total_generated}')
print(f' Output dir: {OUT_DIR}')
if errors:
    print(f' ⚠️  {len(errors)} error(s):')
    for pdf, game, err in errors:
        print(f'    • {pdf} / {game}: {err}')
print(f'============================================================')

In [ ]:
# ── Quality check ────────────────────────────────────────────────────────
import pandas as pd

all_questions = []
for f in sorted(OUT_DIR.glob('*.json')):
    with open(f) as fh:
        qs = json.load(fh)
    for q in qs:
        all_questions.append({
            'file': f.name,
            'game_type': q.get('game_type', '?'),
            'difficulty': q.get('difficulty_teacher', q.get('difficulty', 0)),
            'q_len': len(q.get('question', '')),
            'has_explanation': bool(q.get('explanation', '')),
        })

df = pd.DataFrame(all_questions)
if len(df) == 0:
    print('No questions yet — run the generation cell first')
else:
    print(f'Total questions in rated/: {len(df)}')
    print(f'\nDifficulty distribution:')
    print(df['difficulty'].value_counts().sort_index().rename({1:'easy',2:'medium',3:'hard'}).to_string())
    print(f'\nGame type breakdown:')
    print(df['game_type'].value_counts().to_string())
    print(f'\nAvg question length: {df["q_len"].mean():.0f} chars')
    print(f'Has explanation: {df["has_explanation"].mean()*100:.0f}%')

## Next Steps

1. After generating all PDFs × game types, run **notebook 04** to fine-tune:
   ```
   sbatch ~/ku_prep_arena/ai/scripts/slurm_finetune.sh
   ```
2. The fine-tuned model will be at `ai/models/ku_quiz_v1/`
3. Update `slurm_vllm.sh` to serve the fine-tuned model via LoRA adapters

### Cost estimate (claude-opus-4-6)
- ~15 PDFs × 5 game types = 75 API calls
- ~4K tokens input + 1K output per call ≈ 375K tokens
- Cost ≈ $5/M input + $25/M output ≈ $1.88 + $1.88 ≈ **~$4 total**
